# Creativity Experiment: Baseline vs Steered Paraphrase Consistency Analysis

This notebook tests whether paraphrased versions of the same creativity questions produce consistent feature activations, comparing **baseline (no steering)** vs **steered (creativity-enhanced)** conditions.

We test 7 different creativity tasks, each with 5 paraphrased versions:

1. **Alternate Uses**: Creative ways to use a brick (divergent thinking)
2. **Implications**: Consequences if animals could talk (consequence thinking)
3. **Imagination**: What you'd do if you could fly (personal creativity)
4. **Problem Solving**: Gaining knowledge without books (adaptive thinking)
5. **Creative Planning**: Road trip vs tree house (creative choice)
6. **Product Improvement**: Enhancing a stapler (innovative thinking)
7. **Creative Writing**: Story about a mysterious door (narrative creativity)

**Experiment Design:**
- Each question has 5 semantically equivalent paraphrases
- Each paraphrase is run in **TWO conditions**:
  - **BASELINE**: No neural intervention (natural model behavior)
  - **STEERED**: With creativity-enhancing neural interventions via Goodfire
- Top 10 feature activations are extracted for each paraphrase × condition
- Feature overlap analysis shows which individual features appear consistently across all 5 paraphrases

**Key Questions:**
1. Are neural activations robust to question phrasing?
2. Does creativity steering produce consistent effects across paraphrases?
3. Does steering make activations MORE or LESS consistent?
4. Which individual features are phrasing-independent vs. phrasing-sensitive?

In [1]:
# Clear cache for fresh responses
import shutil
import os

cache_dir = '.edsl_cache'
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

In [2]:
from edsl import QuestionFreeText, QuestionList, QuestionMultipleChoice, Survey, Agent, AgentList, Model
import os
import pandas as pd
import goodfire

In [3]:
# Initialize Goodfire
client = goodfire.Client(os.getenv("GOODFIRE_API_KEY"))
base_variant = goodfire.Variant("meta-llama/Llama-3.3-70B-Instruct")

In [4]:
# Simplified Creativity Experiment Class
class CreativityExperiment:
    def __init__(self, agents, model):
        self.agents = agents
        self.model = model
        self.questions = []
        self.variant = goodfire.Variant("meta-llama/Llama-3.3-70B-Instruct")

    def setup(self, paraphrase_version=0):
        # Multiple creativity tasks for comprehensive divergent thinking assessment
        # paraphrase_version: 0 = original, 1-4 = different paraphrases
        
        # Define all paraphrases for each question
        brick_uses_variants = [
            "List very detailed creative and diverse ways you can use a brick. Each answer should be a paragraph.",
            "Describe multiple imaginative and varied applications for a brick. Provide a full paragraph for each use case.",
            "What are some inventive and unconventional ways a brick could be utilized? Give detailed explanations in paragraph form for each idea.",
            "Think of diverse and original purposes a brick could serve. Write a paragraph describing each potential use.",
            "Generate a variety of creative applications for a standard brick. Each suggestion should be explained thoroughly in a paragraph."
        ]
        
        animals_talk_variants = [
            "What would be the implications if animals could talk? Think broadly and creatively about social, environmental, ethical, and practical consequences.",
            "Imagine if animals gained the ability to communicate verbally with humans. What would the ramifications be across society, nature, ethics, and everyday life?",
            "If animals suddenly developed speech, what effects would this have? Consider the impact on our social structures, the environment, moral frameworks, and practical matters.",
            "Suppose all animals could speak and understand human language. What wide-ranging consequences would follow in terms of society, ecology, ethics, and practical considerations?",
            "What would happen if animals were able to converse with people? Explore the broad implications for social dynamics, environmental concerns, ethical questions, and practical realities."
        ]
        
        flying_ability_variants = [
            "Just suppose you woke up one morning and found you could fly. What would you do? List as many things as you can think of.",
            "Imagine discovering you have the power of flight when you wake up tomorrow. What activities would you pursue? Generate as many ideas as possible.",
            "If you suddenly gained the ability to fly, how would you use this new capability? Enumerate all the things you might do.",
            "Picture yourself waking up with the capacity to soar through the air. What would you choose to do with this ability? List every possibility you can imagine.",
            "Suppose one morning you realized you could fly at will. What actions would you take? Think of as many different things as you can."
        ]
        
        books_disappear_variants = [
            "If all books were to disappear, how would you gain knowledge? Describe creative and diverse methods in detail.",
            "Imagine every book vanished overnight. What alternative ways would you use to acquire knowledge? Explain various inventive approaches thoroughly.",
            "Suppose all written books ceased to exist. How would you learn and gather information? Detail multiple original and varied strategies.",
            "If books were suddenly eliminated from the world, what methods would you employ to obtain knowledge? Describe a range of creative solutions comprehensively.",
            "Consider a scenario where all books disappeared. What diverse and innovative approaches would you take to learning? Provide detailed descriptions of different methods."
        ]
        
        planning_task_variants = [
            "Which task would allow you to be more creative and why?",
            "Which of these activities would give you greater opportunity for creative expression, and what's your reasoning?",
            "Between these two tasks, which one offers more room for creativity? Explain your choice.",
            "Which activity would enable you to exercise more creative thinking, and why do you think so?",
            "Of these options, which would permit the most creative approach? Provide your rationale."
        ]
        
        stapler_improvements_variants = [
            "Your goal is to improve the stapler. List as many specific enhancements as you can that would make it better. You may change features, materials, mechanisms, interfaces, or add/remove parts. Do not list new uses; stay focused on improvements to the object itself. For each idea, add enough detail so someone could build or test it.",
            "How could you make a stapler better? Generate a comprehensive list of detailed upgrades. You can modify its features, materials, working parts, user interface, or components. Keep focus on enhancing the device itself, not finding new applications. Each enhancement should include sufficient detail for implementation.",
            "Think of ways to enhance the design and function of a stapler. List specific improvements with thorough descriptions. Consider changes to materials, mechanisms, features, interfaces, or structural elements. Avoid suggesting new uses; concentrate on making the stapler itself superior. Provide enough detail for each idea to be testable.",
            "What modifications would improve a stapler? Enumerate detailed enhancements covering features, materials, mechanical systems, user interfaces, or component additions/removals. Focus solely on bettering the stapler as an object, not on alternative uses. Each suggestion should be detailed enough for someone to prototype.",
            "Develop a list of concrete improvements for a stapler. Include specific changes to features, materials, mechanisms, interfaces, or parts. Don't propose new applications; stick to enhancing the stapler's existing function. For each improvement, provide enough detail to enable construction or testing."
        ]
        
        creative_story_variants = [
            "Write a creative short story (3-5 paragraphs) based on this prompt: 'A mysterious door appears in your neighborhood that wasn't there yesterday. When you open it...' Be imaginative and original.",
            "Compose an imaginative short narrative (3-5 paragraphs) using this opening: 'An enigmatic door materializes in your community where none existed before. As you turn the handle...' Use creativity and originality.",
            "Craft a creative brief tale (3-5 paragraphs) starting with: 'An unexplained doorway emerges in your local area overnight. Upon opening it...' Be inventive and unique in your storytelling.",
            "Write an original short story (3-5 paragraphs) based on: 'A strange door that wasn't there the day before appears on your street. When you pull it open...' Show imagination and creativity.",
            "Create an imaginative short narrative (3-5 paragraphs) beginning with: 'A puzzling door shows up in your neighborhood where there was none yesterday. The moment you open it...' Be creative and original in your approach."
        ]
        
        # Select the appropriate version based on paraphrase_version (cycle through 0-4)
        version = paraphrase_version % 5
        
        self.questions = [
            # Task 1: Classic divergent thinking - alternate uses
            QuestionList(
                question_name=f"brick_uses_v{version}",
                question_text=brick_uses_variants[version],
                max_list_items=10
            ),
            
            # Task 2: Implications and consequences
            QuestionFreeText(
                question_name=f"animals_talk_v{version}",
                question_text=animals_talk_variants[version]
            ),
            
            # Task 3: Personal imagination - possibilities
            QuestionList(
                question_name=f"flying_ability_v{version}",
                question_text=flying_ability_variants[version],
                max_list_items=15
            ),
            
            # Task 4: Problem-solving creativity
            QuestionFreeText(
                question_name=f"books_disappear_v{version}",
                question_text=books_disappear_variants[version]
            ),
            
            # Task 5: Creative planning choice
            QuestionMultipleChoice(
                question_name=f"planning_task_v{version}",
                question_text=planning_task_variants[version],
                question_options=["Organizing a cross-country road trip", "Building a tree house"]
            ),
            
            # Task 6: Product improvement - detailed enhancements
            QuestionList(
                question_name=f"stapler_improvements_v{version}",
                question_text=stapler_improvements_variants[version],
                max_list_items=12
            ),
            
            # Task 7: Creative writing
            QuestionFreeText(
                question_name=f"creative_story_v{version}",
                question_text=creative_story_variants[version]
            )
        ]

    def run(self, intervention=False):
        if intervention:
            controller = self.get_creativity_intervention()
            self.model.parameters["controller"] = controller
            self.variant = goodfire.Variant("meta-llama/Llama-3.3-70B-Instruct")
            
            if "interventions" in controller:
                for item in controller["interventions"]:
                    if "features" in item and "features" in item["features"]:
                        for feature_data in item["features"]["features"]:
                            features = client.features.search(feature_data["label"], model=self.variant, top_k=1)
                            if features:
                                self.variant.set(features[0], item["value"])
        else:
            self.model.parameters["controller"] = {}
            self.variant = goodfire.Variant("meta-llama/Llama-3.3-70B-Instruct")

        survey = Survey(self.questions)
        return survey.by(self.agents).by(self.model).run(cache=False)

    def get_creativity_intervention(self):
        # NOTE: Replace these with actual feature UUIDs from Goodfire feature search
        return {'interventions': [{'mode': 'nudge',
   'features': {'features': [{'uuid': '2c83bf952a3a4213b45f098aa8c015e2',
      'label': 'Enabling or empowering creative expression and exploration',
      'index_in_sae': 13142,
      'max_activation_strength': 1}]},
   'value': 0.5}],
 'scopes': [],
 'name': 'controller__45610380',
 'nonzero_strength_threshold': None,
 'min_nudge_entropy': None,
 'max_nudge_entropy': None}

    def analyze_features(self, results, condition_name, use_base=False, question_name=None):
        """Analyze and return feature activations for a question."""
        variant = base_variant if use_base else self.variant
        
        # If no specific question name, use the first question
        if question_name is None:
            question_name = self.questions[0].question_name
        
        try:
            prompts = results.select(f"prompt.{question_name}_user_prompt").to_list()
            responses = results.select(f"generated_tokens.{question_name}_generated_tokens").to_list()
            
            if prompts and responses:
                prompt = prompts[0]["text"] if isinstance(prompts[0], dict) else prompts[0]
                response = responses[0]
                
                inspector = client.features.inspect(
                    [{"role": "user", "content": str(prompt)}, {"role": "assistant", "content": str(response)}],
                    model=variant
                )
                
                # Return top features as a list of dicts
                return [{"label": act.feature.label, "activation": act.activation} 
                        for act in inspector.top(k=10)]
        except Exception as e:
            return [{"error": str(e)}]
    
    def get_response_data(self, results, question_name):
        """Extract answer and comment for a question."""
        answer = results.select(f"answer.{question_name}").to_list()
        comment = results.select(f"comment.{question_name}_comment").to_list()
        return {
            "answer": answer[0] if answer else "No response",
            "comment": comment[0] if comment else "No comment"
        }

In [5]:
# Preview all paraphrase variants before running
print("="*80)
print("PREVIEW OF ALL QUESTION PARAPHRASES")
print("="*80)

question_variants = {
    "Brick Uses": [
        "List very detailed creative and diverse ways you can use a brick. Each answer should be a paragraph.",
        "Describe multiple imaginative and varied applications for a brick. Provide a full paragraph for each use case.",
        "What are some inventive and unconventional ways a brick could be utilized? Give detailed explanations in paragraph form for each idea.",
        "Think of diverse and original purposes a brick could serve. Write a paragraph describing each potential use.",
        "Generate a variety of creative applications for a standard brick. Each suggestion should be explained thoroughly in a paragraph."
    ],
    "Animals Talk": [
        "What would be the implications if animals could talk? Think broadly and creatively about social, environmental, ethical, and practical consequences.",
        "Imagine if animals gained the ability to communicate verbally with humans. What would the ramifications be across society, nature, ethics, and everyday life?",
        "If animals suddenly developed speech, what effects would this have? Consider the impact on our social structures, the environment, moral frameworks, and practical matters.",
        "Suppose all animals could speak and understand human language. What wide-ranging consequences would follow in terms of society, ecology, ethics, and practical considerations?",
        "What would happen if animals were able to converse with people? Explore the broad implications for social dynamics, environmental concerns, ethical questions, and practical realities."
    ],
    "Flying Ability": [
        "Just suppose you woke up one morning and found you could fly. What would you do? List as many things as you can think of.",
        "Imagine discovering you have the power of flight when you wake up tomorrow. What activities would you pursue? Generate as many ideas as possible.",
        "If you suddenly gained the ability to fly, how would you use this new capability? Enumerate all the things you might do.",
        "Picture yourself waking up with the capacity to soar through the air. What would you choose to do with this ability? List every possibility you can imagine.",
        "Suppose one morning you realized you could fly at will. What actions would you take? Think of as many different things as you can."
    ],
    "Books Disappear": [
        "If all books were to disappear, how would you gain knowledge? Describe creative and diverse methods in detail.",
        "Imagine every book vanished overnight. What alternative ways would you use to acquire knowledge? Explain various inventive approaches thoroughly.",
        "Suppose all written books ceased to exist. How would you learn and gather information? Detail multiple original and varied strategies.",
        "If books were suddenly eliminated from the world, what methods would you employ to obtain knowledge? Describe a range of creative solutions comprehensively.",
        "Consider a scenario where all books disappeared. What diverse and innovative approaches would you take to learning? Provide detailed descriptions of different methods."
    ],
    "Planning Task": [
        "Which task would allow you to be more creative and why?",
        "Which of these activities would give you greater opportunity for creative expression, and what's your reasoning?",
        "Between these two tasks, which one offers more room for creativity? Explain your choice.",
        "Which activity would enable you to exercise more creative thinking, and why do you think so?",
        "Of these options, which would permit the most creative approach? Provide your rationale."
    ],
    "Stapler Improvements": [
        "Your goal is to improve the stapler. List as many specific enhancements as you can that would make it better. You may change features, materials, mechanisms, interfaces, or add/remove parts. Do not list new uses; stay focused on improvements to the object itself. For each idea, add enough detail so someone could build or test it.",
        "How could you make a stapler better? Generate a comprehensive list of detailed upgrades. You can modify its features, materials, working parts, user interface, or components. Keep focus on enhancing the device itself, not finding new applications. Each enhancement should include sufficient detail for implementation.",
        "Think of ways to enhance the design and function of a stapler. List specific improvements with thorough descriptions. Consider changes to materials, mechanisms, features, interfaces, or structural elements. Avoid suggesting new uses; concentrate on making the stapler itself superior. Provide enough detail for each idea to be testable.",
        "What modifications would improve a stapler? Enumerate detailed enhancements covering features, materials, mechanical systems, user interfaces, or component additions/removals. Focus solely on bettering the stapler as an object, not on alternative uses. Each suggestion should be detailed enough for someone to prototype.",
        "Develop a list of concrete improvements for a stapler. Include specific changes to features, materials, mechanisms, interfaces, or parts. Don't propose new applications; stick to enhancing the stapler's existing function. For each improvement, provide enough detail to enable construction or testing."
    ],
    "Creative Story": [
        "Write a creative short story (3-5 paragraphs) based on this prompt: 'A mysterious door appears in your neighborhood that wasn't there yesterday. When you open it...' Be imaginative and original.",
        "Compose an imaginative short narrative (3-5 paragraphs) using this opening: 'An enigmatic door materializes in your community where none existed before. As you turn the handle...' Use creativity and originality.",
        "Craft a creative brief tale (3-5 paragraphs) starting with: 'An unexplained doorway emerges in your local area overnight. Upon opening it...' Be inventive and unique in your storytelling.",
        "Write an original short story (3-5 paragraphs) based on: 'A strange door that wasn't there the day before appears on your street. When you pull it open...' Show imagination and creativity.",
        "Create an imaginative short narrative (3-5 paragraphs) beginning with: 'A puzzling door shows up in your neighborhood where there was none yesterday. The moment you open it...' Be creative and original in your approach."
    ]
}

for task_name, variants in question_variants.items():
    print(f"\n{'='*80}")
    print(f"TASK: {task_name.upper()}")
    print('='*80)
    for i, variant in enumerate(variants):
        print(f"\nVersion {i}:")
        print(f"  {variant}")
    print()

print("="*80)
print("Ready to run the experiment with all paraphrase variants!")
print("="*80)


PREVIEW OF ALL QUESTION PARAPHRASES

TASK: BRICK USES

Version 0:
  List very detailed creative and diverse ways you can use a brick. Each answer should be a paragraph.

Version 1:
  Describe multiple imaginative and varied applications for a brick. Provide a full paragraph for each use case.

Version 2:
  What are some inventive and unconventional ways a brick could be utilized? Give detailed explanations in paragraph form for each idea.

Version 3:
  Think of diverse and original purposes a brick could serve. Write a paragraph describing each potential use.

Version 4:
  Generate a variety of creative applications for a standard brick. Each suggestion should be explained thoroughly in a paragraph.


TASK: ANIMALS TALK

Version 0:
  What would be the implications if animals could talk? Think broadly and creatively about social, environmental, ethical, and practical consequences.

Version 1:
  Imagine if animals gained the ability to communicate verbally with humans. What would the ram

In [6]:
# Create agents and model
agents = AgentList([Agent(name=f"Agent_{i}", traits={"id": i}) for i in range(1, 2)])
model = Model("meta-llama/Llama-3.3-70B-Instruct", service_name="goodfire")

In [7]:
# Run experiment with all 5 paraphrase versions (BOTH steered and non-steered)
timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
results_dir = f"creativity_experiment/results_paraphrase_{timestamp}"
os.makedirs(results_dir, exist_ok=True)

print("\n" + "="*80)
print("RUNNING PARAPHRASE COMPARISON EXPERIMENT")
print("Testing 5 paraphrased versions - BOTH Steered and Non-Steered")
print("="*80)

# Store all results for comparison
all_results = {
    'steered': {},
    'baseline': {}
}
all_features = {
    'steered': {},
    'baseline': {}
}

base_questions = ["brick_uses", "animals_talk", "flying_ability", "books_disappear", 
                 "planning_task", "stapler_improvements", "creative_story"]

# Run each paraphrase version
for version in range(5):
    print(f"\n{'='*80}")
    print(f"PARAPHRASE VERSION {version}")
    print(f"{'='*80}")
    
    # Run BASELINE (no steering) for this paraphrase
    print(f"\n⏳ Running Version {version} - BASELINE (No Steering)...")
    baseline_model = Model("meta-llama/Llama-3.3-70B-Instruct", service_name="goodfire")
    baseline_exp = CreativityExperiment(agents, baseline_model)
    baseline_exp.setup(paraphrase_version=version)
    baseline_results = baseline_exp.run(intervention=False)
    baseline_results.to_csv(f"{results_dir}/baseline_v{version}.csv")
    
    all_results['baseline'][version] = baseline_results
    all_features['baseline'][version] = {}
    
    for base_q in base_questions:
        q_name = f"{base_q}_v{version}"
        features = baseline_exp.analyze_features(baseline_results, f"Baseline_V{version}", use_base=True, question_name=q_name)
        all_features['baseline'][version][base_q] = features
    
    print(f"✓ Baseline Version {version} complete")
    
    # Run STEERED (with creativity intervention) for this paraphrase
    print(f"\n⏳ Running Version {version} - STEERED (Creativity Enhancement)...")
    steered_model = Model("meta-llama/Llama-3.3-70B-Instruct", service_name="goodfire")
    steered_exp = CreativityExperiment(agents, steered_model)
    steered_exp.setup(paraphrase_version=version)
    steered_results = steered_exp.run(intervention=True)
    steered_results.to_csv(f"{results_dir}/steered_v{version}.csv")
    
    all_results['steered'][version] = steered_results
    all_features['steered'][version] = {}
    
    for base_q in base_questions:
        q_name = f"{base_q}_v{version}"
        features = steered_exp.analyze_features(steered_results, f"Steered_V{version}", use_base=False, question_name=q_name)
        all_features['steered'][version][base_q] = features
    
    print(f"✓ Steered Version {version} complete")

# Display activation comparison across paraphrases - BOTH BASELINE AND STEERED
print("\n\n" + "="*80)
print("TOP 10 FEATURE CONSISTENCY ACROSS PARAPHRASES")
print("Comparing BASELINE vs STEERED")
print("="*80)

for base_q in base_questions:
    print("\n\n" + "█"*80)
    print(f"  TASK: {base_q.upper().replace('_', ' ')}")
    print("█"*80)
    
    # Show each paraphrase's question text
    print("\n📋 Question Paraphrases:")
    for v in range(5):
        model = Model("meta-llama/Llama-3.3-70B-Instruct", service_name="goodfire")
        exp = CreativityExperiment(agents, model)
        exp.setup(paraphrase_version=v)
        q_obj = [q for q in exp.questions if base_q in q.question_name][0]
        print(f"\n  V{v}: {q_obj.question_text[:100]}...")
    
    # Analyze both baseline and steered
    for condition in ['baseline', 'steered']:
        condition_label = "BASELINE (No Steering)" if condition == 'baseline' else "STEERED (Creativity Enhanced)"
        print(f"\n\n{'='*80}")
        print(f"🔷 {condition_label}")
        print("="*80)
        
        # Show top 10 features for each paraphrase
        print("\n🧠 TOP 10 FEATURES FOR EACH PARAPHRASE:")
        print("─"*80)
        
        for v in range(5):
            print(f"\n  {'Version ' + str(v) + ' Top 10':.<80}")
            features = all_features[condition][v][base_q]
            for i, feat in enumerate(features[:10], 1):
                if "error" in feat:
                    print(f"    {i:2d}. Error: {feat['error']}")
                else:
                    print(f"    {i:2d}. [{feat['activation']:>6.1f}] {feat['label'][:70]}")
        
        # Calculate feature overlap
        print("\n\n📊 FEATURE OVERLAP ANALYSIS:")
        print("─"*80)
        
        feature_appearances = {}
        
        for v in range(5):
            features = all_features[condition][v][base_q]
            for rank, feat in enumerate(features[:10], 1):
                if "error" not in feat:
                    label = feat['label']
                    if label not in feature_appearances:
                        feature_appearances[label] = []
                    feature_appearances[label].append((v, rank, feat['activation']))
        
        sorted_features = sorted(feature_appearances.items(), 
                                key=lambda x: len(x[1]), 
                                reverse=True)
        
        print(f"\n  Features appearing in ALL 5 paraphrases:")
        all_five = [f for f, appearances in sorted_features if len(appearances) == 5]
        if all_five:
            for feat_label in all_five:
                appearances = feature_appearances[feat_label]
                ranks = [rank for v, rank, act in appearances]
                avg_rank = sum(ranks) / len(ranks)
                activations = [act for v, rank, act in appearances]
                avg_act = sum(activations) / len(activations)
                print(f"    ✓ {feat_label[:65]}")
                print(f"      Ranks: {ranks}, Avg: {avg_rank:.1f} | Activations: {[f'{a:.1f}' for a in activations]}, Avg: {avg_act:.1f}")
        else:
            print(f"    ⚠️  No features appear in all 5 paraphrases")
        
        print(f"\n  Features appearing in 4/5 paraphrases:")
        four_of_five = [f for f, appearances in sorted_features if len(appearances) == 4]
        if four_of_five:
            for feat_label in four_of_five[:5]:
                appearances = feature_appearances[feat_label]
                versions = [v for v, rank, act in appearances]
                ranks = [rank for v, rank, act in appearances]
                print(f"    • {feat_label[:65]}")
                print(f"      In versions: {versions}, Ranks: {ranks}")
        else:
            print(f"    None")
        
        print(f"\n  Features appearing in 3/5 paraphrases:")
        three_of_five = [f for f, appearances in sorted_features if len(appearances) == 3]
        if three_of_five:
            for feat_label in three_of_five[:5]:
                appearances = feature_appearances[feat_label]
                versions = [v for v, rank, act in appearances]
                print(f"    • {feat_label[:65]}")
                print(f"      In versions: {versions}")
        else:
            print(f"    None")
        
        # Summary statistics
        total_unique_features = len(feature_appearances)
        features_in_all = len([f for f, app in sorted_features if len(app) == 5])
        features_in_majority = len([f for f, app in sorted_features if len(app) >= 3])
        
        print(f"\n  Summary:")
        print(f"    Total unique features: {total_unique_features}")
        print(f"    In ALL 5 versions: {features_in_all}")
        print(f"    In 3+ versions: {features_in_majority}")
        
        if features_in_all >= 5:
            print(f"    ✓ HIGH CONSISTENCY")
        elif features_in_all >= 3:
            print(f"    ~ MODERATE CONSISTENCY")
        else:
            print(f"    ⚠️  LOW CONSISTENCY")

print("\n\n" + "="*80)
print(f"✓ All results saved to: {results_dir}")
print("="*80)


RUNNING PARAPHRASE COMPARISON EXPERIMENT
Testing 5 paraphrased versions - BOTH Steered and Non-Steered

PARAPHRASE VERSION 0

⏳ Running Version 0 - BASELINE (No Steering)...
✓ Baseline Version 0 complete

⏳ Running Version 0 - STEERED (Creativity Enhancement)...
✓ Steered Version 0 complete

PARAPHRASE VERSION 1

⏳ Running Version 1 - BASELINE (No Steering)...
✓ Baseline Version 1 complete

⏳ Running Version 1 - STEERED (Creativity Enhancement)...
✓ Steered Version 1 complete

PARAPHRASE VERSION 2

⏳ Running Version 2 - BASELINE (No Steering)...
✓ Baseline Version 2 complete

⏳ Running Version 2 - STEERED (Creativity Enhancement)...
✓ Steered Version 2 complete

PARAPHRASE VERSION 3

⏳ Running Version 3 - BASELINE (No Steering)...
✓ Baseline Version 3 complete

⏳ Running Version 3 - STEERED (Creativity Enhancement)...
✓ Steered Version 3 complete

PARAPHRASE VERSION 4

⏳ Running Version 4 - BASELINE (No Steering)...
✓ Baseline Version 4 complete

⏳ Running Version 4 - STEERED (Creativ

In [8]:
# Summary Analysis: Compare Baseline vs Steered Feature Consistency
print("\n\n" + "="*80)
print("CROSS-TASK SUMMARY: BASELINE vs STEERED FEATURE CONSISTENCY")
print("="*80)

# Analyze both conditions
for condition in ['baseline', 'steered']:
    condition_label = "BASELINE (No Steering)" if condition == 'baseline' else "STEERED (Creativity Enhanced)"
    print(f"\n{'='*80}")
    print(f"📊 {condition_label}")
    print("="*80)
    
    summary_data = []
    
    for base_q in base_questions:
        # Calculate feature overlap for this task
        feature_appearances = {}
        
        for v in range(5):
            features = all_features[condition][v][base_q]
            for rank, feat in enumerate(features[:10], 1):
                if "error" not in feat:
                    label = feat['label']
                    if label not in feature_appearances:
                        feature_appearances[label] = []
                    feature_appearances[label].append((v, rank, feat['activation']))
        
        total_unique = len(feature_appearances)
        features_in_all_5 = len([f for f, app in feature_appearances.items() if len(app) == 5])
        features_in_4_plus = len([f for f, app in feature_appearances.items() if len(app) >= 4])
        features_in_3_plus = len([f for f, app in feature_appearances.items() if len(app) >= 3])
        
        avg_overlap_pct = (features_in_all_5 / 10.0) * 100 if features_in_all_5 > 0 else 0
        
        summary_data.append({
            'Task': base_q.replace('_', ' ').title(),
            'Total Unique': total_unique,
            'In All 5': features_in_all_5,
            'In 4+': features_in_4_plus,
            'In 3+': features_in_3_plus,
            'Overlap %': f"{avg_overlap_pct:.0f}%"
        })
    
    summary_df = pd.DataFrame(summary_data)
    print("\n")
    print(summary_df.to_string(index=False))
    
    # Statistics for this condition
    print(f"\n  KEY INSIGHTS:")
    
    consistent_tasks = [(row['Task'], row['In All 5']) for _, row in summary_df.iterrows()]
    consistent_tasks.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\n  Most consistent tasks (features in all 5 paraphrases):")
    for i, (task, count) in enumerate(consistent_tasks[:3], 1):
        print(f"    {i}. {task}: {count} features")
    
    # Overall statistics
    avg_features_all_5 = sum(row['In All 5'] for _, row in summary_df.iterrows()) / len(summary_df)
    avg_unique = sum(row['Total Unique'] for _, row in summary_df.iterrows()) / len(summary_df)
    
    print(f"\n  Overall:")
    print(f"    Avg features in ALL 5 paraphrases: {avg_features_all_5:.1f}")
    print(f"    Avg total unique features: {avg_unique:.1f}")
    print(f"    Consistency ratio: {(avg_features_all_5 / 10) * 100:.0f}%")
    
    if avg_features_all_5 >= 5:
        print(f"    ✓ HIGH CONSISTENCY")
    elif avg_features_all_5 >= 3:
        print(f"    ~ MODERATE CONSISTENCY")
    else:
        print(f"    ⚠️  LOW CONSISTENCY")

# Compare baseline vs steered
print(f"\n\n{'='*80}")
print("🔍 BASELINE vs STEERED COMPARISON")
print("="*80)

print("\nKey Questions:")
print("1. Do BASELINE and STEERED show similar consistency levels?")
print("2. Does steering make activations MORE or LESS consistent across paraphrases?")
print("3. Are there specific features that appear consistently in STEERED but not BASELINE?")

# Calculate comparative metrics
baseline_avg_consistency = []
steered_avg_consistency = []

for base_q in base_questions:
    # Baseline
    baseline_features = {}
    for v in range(5):
        for feat in all_features['baseline'][v][base_q][:10]:
            if "error" not in feat:
                baseline_features[feat['label']] = baseline_features.get(feat['label'], 0) + 1
    baseline_in_all = len([f for f, count in baseline_features.items() if count == 5])
    baseline_avg_consistency.append(baseline_in_all)
    
    # Steered
    steered_features = {}
    for v in range(5):
        for feat in all_features['steered'][v][base_q][:10]:
            if "error" not in feat:
                steered_features[feat['label']] = steered_features.get(feat['label'], 0) + 1
    steered_in_all = len([f for f, count in steered_features.items() if count == 5])
    steered_avg_consistency.append(steered_in_all)

baseline_avg = sum(baseline_avg_consistency) / len(baseline_avg_consistency)
steered_avg = sum(steered_avg_consistency) / len(steered_avg_consistency)

print(f"\nOverall Consistency Comparison:")
print(f"  BASELINE average: {baseline_avg:.1f} features in all 5 paraphrases")
print(f"  STEERED average:  {steered_avg:.1f} features in all 5 paraphrases")
print(f"  Difference:       {steered_avg - baseline_avg:+.1f}")

if abs(steered_avg - baseline_avg) < 1:
    print("\n  → Similar consistency levels - steering doesn't significantly affect robustness")
elif steered_avg > baseline_avg:
    print("\n  → STEERED is MORE consistent - steering produces more robust activations!")
else:
    print("\n  → BASELINE is MORE consistent - steering introduces more variability")

print("\n" + "="*80)




CROSS-TASK SUMMARY: BASELINE vs STEERED FEATURE CONSISTENCY

📊 BASELINE (No Steering)


                Task  Total Unique  In All 5  In 4+  In 3+ Overlap %
          Brick Uses            19         6      7      7       60%
        Animals Talk            14         6      9     10       60%
      Flying Ability            16         6      7      8       60%
     Books Disappear            18         4      7      9       40%
       Planning Task            23         5      5      6       50%
Stapler Improvements            14         6      8     11       60%
      Creative Story            29         1      3      6       10%

  KEY INSIGHTS:

  Most consistent tasks (features in all 5 paraphrases):
    1. Brick Uses: 6 features
    2. Animals Talk: 6 features
    3. Flying Ability: 6 features

  Overall:
    Avg features in ALL 5 paraphrases: 4.9
    Avg total unique features: 19.0
    Consistency ratio: 49%
    ~ MODERATE CONSISTENCY

📊 STEERED (Creativity Enhanced)


       

## How to Interpret the Results

### What This Experiment Tests
This notebook runs **5 paraphrased versions** of each creativity question with **BOTH baseline and steered conditions** to test:
1. **Which individual neural features appear in the top 10** regardless of phrasing (for each condition)
2. **Whether baseline and steered show different consistency patterns**
3. **If steering makes activations more or less robust to phrasing variations**

### Key Metrics to Look For

**Individual Feature Overlap (for each condition):**
- **Features in ALL 5 versions** = Core features that always activate for this task type
- **Features in 4/5 versions** = Mostly consistent, minor phrasing sensitivity
- **Features in 3/5 versions** = Moderate consistency
- **Total Unique Features** = Shows variability (lower is more consistent)

**Consistency Comparison:**
- **Baseline consistency** = How robust are natural activations to phrasing?
- **Steered consistency** = How robust are creativity-enhanced activations to phrasing?
- **Difference** = Does steering increase or decrease robustness?

### What Good Results Look Like

**For individual conditions (baseline or steered):**
- **5+ features appearing in ALL 5 paraphrases** - robust core features
- **Low total unique features** (ideally 15-20) - low variability
- **Similar feature ranks** across versions

**For the comparison:**
- **Similar consistency in both conditions** = Steering doesn't affect robustness
- **Higher consistency in steered** = Steering produces more canonical/robust activations
- **Higher consistency in baseline** = Steering introduces more variability (potentially concerning)

### What the Analysis Shows You

**For each task, both baseline and steered:**
1. **Top 10 features for each paraphrase** - See what activates in each version
2. **Feature overlap analysis** - Which features are consistent vs. variable
3. **Summary statistics** - Overall consistency metrics

**In the cross-task summary:**
- Consistency comparison table for baseline
- Consistency comparison table for steered
- **Direct comparison**: Which condition is more robust to phrasing?

### Key Questions Answered

1. **Are neural activations robust to paraphrasing?**
   - Look at "In All 5" counts - higher = more robust

2. **Does creativity steering produce consistent effects?**
   - Compare baseline vs steered consistency
   - Check if specific creativity features appear reliably

3. **Which tasks are most/least sensitive to phrasing?**
   - Compare across the 7 creativity tasks
   - Identify which task types have stable vs. variable features

4. **Are steered activations more predictable?**
   - If steered shows higher consistency, steering creates more canonical responses
   - If baseline shows higher consistency, steering may be adding noise

### Interpretation Guidelines

**HIGH consistency in both baseline and steered (5+ features):**
- ✓ Neural representations are robust
- ✓ Semantically equivalent questions trigger the same circuits
- ✓ Steering works on stable underlying features

**HIGH baseline, LOW steered consistency:**
- ⚠️ Steering may be introducing variability
- Could indicate steering is phrasing-sensitive
- May need to adjust steering parameters

**LOW baseline, HIGH steered consistency:**
- ✓ Steering actually IMPROVES robustness
- Steering creates more canonical activation patterns
- Excellent sign for mechanistic interpretability

**LOW consistency in both:**
- ⚠️ Task may be inherently phrasing-sensitive
- Could indicate these question types are less well-defined
- May need different paraphrasing strategies
